## Lab 11: Embeddings and Vision Transformer
*Suggested time: 40-45 minutes*

Compute: **CPU**

We show demonstrate the following,

- A. Image and Audio  Embeddings
- B. Vision Transformer (ViT) and Comparision of CNN (ResNet18) and ViT model


## **A. Image and Audio Embeddings**

##### **Step 0**:

- Get embeddings from MobileNet_v3 model
- Upload image files for experimentation

In [ ]:
!wget -O embedder.tflite -q https://storage.googleapis.com/mediapipe-models/image_embedder/mobilenet_v3_small/float32/1/mobilenet_v3_small.tflite

In [ ]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/images.zip
!unzip -q images.zip

##### **Step 1:** Preparing Images
Now that we have two images that will be compared, we can display them to confirm that they look correct. For this example you see two separate, but similar images of cats.

In [ ]:
import cv2
import math
import matplotlib.pyplot as plt
import os


BASE_DIR = os.getcwd()

IMAGE_FILENAMES = ['cat.png', 'cat.png']


DESIRED_HEIGHT = 480
DESIRED_WIDTH = 480

def cv2_imshow(img):
  plt.figure(figsize=(8, 8))
  if len(img.shape) == 2:
    plt.imshow(img, cmap='gray')
  else:
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()

def resize_and_show(image):
  if image is None:
    print("Could not load image.")
    return

  h, w = image.shape[:2]
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h / (w / DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w / (h / DESIRED_HEIGHT)), DESIRED_HEIGHT))
  cv2_imshow(img)


# Preview the images.
for name in IMAGE_FILENAMES:
  full_path = os.path.join(BASE_DIR, 'images', name)
  print(full_path)
  image = cv2.imread(full_path)
  resize_and_show(image)

##### **Step 2:** Create and Compare Image Embeddings

Once everything looks good, we can start comparing embeddings of both the images. We will start by creating the options that are necessary for associating the model with the Image Embedder, as well as some customizations.

Next we create the Image Embedder, then format your two images for MediaPipe so that you can use cosine similarity to compare them. Finally, we display the similarity value.

In [ ]:
!pip install mediapipe

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os

# Create options for Image Embedder
base_options = python.BaseOptions(model_asset_path='embedder.tflite')
l2_normalize = True
quantize = True
options = vision.ImageEmbedderOptions(
    base_options=base_options, l2_normalize=l2_normalize, quantize=quantize)

# Use full image paths
first_path = os.path.join(BASE_DIR, 'images', IMAGE_FILENAMES[0])
second_path = os.path.join(BASE_DIR, 'images', IMAGE_FILENAMES[1])

# Create Image Embedder
with vision.ImageEmbedder.create_from_options(options) as embedder:
  first_image = mp.Image.create_from_file(first_path)
  second_image = mp.Image.create_from_file(second_path)

  first_embedding_result = embedder.embed(first_image)
  second_embedding_result = embedder.embed(second_image)

  similarity = vision.ImageEmbedder.cosine_similarity(
      first_embedding_result.embeddings[0],
      second_embedding_result.embeddings[0])
  print(similarity)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import urllib

# Extract the embeddings as NumPy arrays for burger images
embedding1_np = np.array(first_embedding_result.embeddings[0].embedding)
embedding2_np = np.array(second_embedding_result.embeddings[0].embedding)

all_embeddings_list = [embedding1_np, embedding2_np]
labels_for_plot = ['Image 1 ', 'Image 2 ']
plot_markers = ['o', 'o']

combined_embeddings = np.vstack(all_embeddings_list)

print(f"Embedding 1 Dimensions : {embedding1_np.shape}")
print(f"Embedding 2 Dimensions: {embedding2_np.shape}")

# Initialize PCA to reduce to 2 components
pca_vision = PCA(n_components=2)

# Fit PCA to the combined embeddings and transform them
pca_result_vision = pca_vision.fit_transform(combined_embeddings)

# Create a scatter plot of the PCA-transformed embeddings
plt.figure(figsize=(10, 8))
for i in range(len(all_embeddings_list)):
    plt.scatter(pca_result_vision[i, 0], pca_result_vision[i, 1], label=labels_for_plot[i], s=100, marker=plot_markers[i])

plt.title('Vision Embeddings PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.legend()
plt.show()

print(f"PCA-transformed vision embedding shape: {pca_result_vision.shape}")
print(f"Explained variance ratio by principal components: {pca_vision.explained_variance_ratio_}")

**Experiments with Image Embeddings:**

- Check cosine similarity and 2-dimensional PCA embeddings of the same image
- Verify cosine similarity of images of your choice - similar or very different
- Work with multiple images and see how cosine similarity and PCA embeddings are presented

**Audio Embeddings**

##### **Step 3:**

Review the inputs and outputs of the YAMNet acoustic model

- **Input format**
  - Accepts a **1-D float32 Tensor or NumPy array**.
  - The waveform can be **any length**.
  - Audio must be **mono, 16 kHz**, and scaled to the range **[-1.0, +1.0]**.

- **Frame size and overlap**
  - The waveform is split into **0.96-second frames**.
  - Frames overlap with a **0.48-second hop**.
  - The model then processes all framed segments in a batch.

- **Returned outputs**
  - The model returns a **3 value tuple**:
    - `scores`
    - `embeddings`
    - `log_mel_spectrogram`

- **`scores` output**
  - Shape: **(N, 521)**
  - `N` is the number of framed audio windows.
  - Contains **per-frame prediction scores** for the **521 supported AudioSet classes**.
  - Can be used to detect audio events by aggregating across frames, such as with **mean** or **max**.

- **`embeddings` output**
  - Shape: **(N, 1024)**
  - Contains **per-frame embedding vectors**.
  - These embeddings are the **average-pooled features** used before the final classifier layer.
  - Useful for:
    - building larger models
    - using YAMNet as a **feature extractor**
    - training shallow downstream classifiers

- **`log_mel_spectrogram` output**
  - Represents the **log mel spectrogram** of the full waveform.
  - Shape: **(num_spectrogram_frames, 64)**
  - Computed using:
    - **0.025-second analysis windows**
    - **0.01-second hop**
    - **64 mel bins**
  - Mainly useful for **visualization** and **debugging**.

- **Class mapping**
  - Each column in `scores` corresponds to an **AudioSet class**.
  - Column indices **0–520** map to class names using the **YAMNet Class Map**.
  - The class map is available:
    - as a **CSV file** in the GitHub repository


##### **Step 4**:  Install TensorFlow Hub

In [ ]:
!pip install -q tensorflow_hub

##### **Step 5**:  Assert shapes of scores, embeddings and spectrograms of YAMNet model

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import csv
import io

# Load the model.
model = hub.load('https://www.kaggle.com/models/google/yamnet/TensorFlow2/yamnet/1')

# Input: 3 seconds of silence as mono 16 kHz waveform samples.
waveform = np.zeros(3 * 16000, dtype=np.float32)

# Run the model, check the output.
scores, embeddings, log_mel_spectrogram = model(waveform)
scores.shape.assert_is_compatible_with([None, 521])
embeddings.shape.assert_is_compatible_with([None, 1024])
log_mel_spectrogram.shape.assert_is_compatible_with([None, 64])

# Print the embeddings vector as a NumPy array
print(embeddings.numpy().shape)
print (scores.numpy().shape)
print ('Spectrogram shape', log_mel_spectrogram.numpy().shape)
print ('Spectrogram values', log_mel_spectrogram.numpy())

##### **Step 6**:  Visualize YAMNet Embeddings with PCA

- PCA is used to reduce YAMNet’s 1024-dimensional embeddings to 2 or 3 dimensions.
- This dimensionality reduction makes the embeddings easier to visualize on 2D or 3D plots.
- Visualizing the embeddings can help reveal clusters or patterns in the audio representation space.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Ensure embeddings are a NumPy array
embeddings_np = embeddings.numpy()

# Initialize PCA to reduce to 2 components for 2D plotting
pca = PCA(n_components=2)

# Fit PCA to the embeddings and transform them
# Each row in embeddings_np corresponds to an audio frame's embedding
pca_result = pca.fit_transform(embeddings_np)

# Create a scatter plot of the PCA-transformed embeddings
plt.figure(figsize=(10, 8))
plt.scatter(pca_result[:, 0], pca_result[:, 1], alpha=0.7)
plt.title('YAMNet Embeddings PCA (Silence Input)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
# Removed xlim and ylim to zoom out
plt.show()

print(f"Original embedding shape: {embeddings_np.shape}")
print(f"PCA-transformed embedding shape: {pca_result.shape}")
print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_}")

##### **Step 7**: Generate YAMNet Embeddings for Multiple Sound Waves and Visualize with PCA

This section demonstrates how to generate embeddings for different types of sound waves,
- silence,
- sine wave of 440 Hz
- random noise
- Square wave - 220 Hz
- Sawtooth wave - 100 Hz

using the YAMNet model, and then visualize these high-dimensional embeddings in a 2D space using Principal Component Analysis (PCA). This helps in understanding how YAMNet distinguishes between different audio patterns.

In [ ]:
import numpy as np
import tensorflow as tf

# Generate different waveforms
sample_rate = 16000 # YAMNet expects 16 kHz audio
duration = 3 # seconds

# 1. Silence
waveform_silence = np.zeros(int(duration * sample_rate), dtype=np.float32)

# 2. Sine wave (e.g., 440 Hz tone)
frequency_sine = 440 # Hz
t = np.linspace(0, duration, int(duration * sample_rate), endpoint=False)
waveform_sine = 0.5 * np.sin(2 * np.pi * frequency_sine * t).astype(np.float32)

# 3. Random noise
waveform_noise = np.random.uniform(-0.5, 0.5, int(duration * sample_rate)).astype(np.float32)

# 4. Square wave (e.g., 220 Hz)
frequency_square = 220 # Hz
waveform_square = 0.5 * np.sign(np.sin(2 * np.pi * frequency_square * t)).astype(np.float32)

# 5. Sawtooth wave (e.g., 100 Hz)
frequency_sawtooth = 100 # Hz
waveform_sawtooth = 0.5 * (2 * (t * frequency_sawtooth - np.floor(t * frequency_sawtooth + 0.5))).astype(np.float32)

waveforms = {
    "Silence": waveform_silence,
    "Sine Wave (440 Hz)": waveform_sine,
    "Random Noise": waveform_noise,
    "Square Wave (220 Hz)": waveform_square,
    "Sawtooth Wave (100 Hz)": waveform_sawtooth
}

print("Generated waveforms:")
for name, wf in waveforms.items():
    print(f"- {name}: {wf.shape} samples")

In [ ]:
all_embeddings = []
embedding_labels = []

# Ensure the YAMNet model is loaded
# If model is not defined, load it here
if 'model' not in locals():
    import tensorflow_hub as hub
    model = hub.load('https://www.kaggle.com/models/google/yamnet/TensorFlow2/yamnet/1')


for label, waveform in waveforms.items():
    # YAMNet can handle arbitrary length, but internally frames it.
    # It returns one embedding per frame, so we'll average them.
    scores, embeddings, log_mel_spectrogram = model(waveform)

    # Take the mean of embeddings if YAMNet returns multiple frames
    if embeddings.shape[0] > 1:
        averaged_embedding = np.mean(embeddings.numpy(), axis=0)
    else:
        averaged_embedding = embeddings.numpy()[0]

    all_embeddings.append(averaged_embedding)
    embedding_labels.append(label)

embeddings_array = np.array(all_embeddings)

print(f"Collected {len(all_embeddings)} embeddings, each of shape {embeddings_array.shape[1]}")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Initialize PCA to reduce to 2 components
pca_multi = PCA(n_components=2)

# Fit PCA to the collected embeddings and transform them
pca_result_multi = pca_multi.fit_transform(embeddings_array)

# Create a scatter plot of the PCA-transformed embeddings
plt.figure(figsize=(10, 8))
for i, (x, y) in enumerate(pca_result_multi):
    plt.scatter(x, y, label=embedding_labels[i], s=100) # s is marker size
    plt.annotate(embedding_labels[i], (x + 0.05, y + 0.05)) # Add label next to point

plt.title('YAMNet Embeddings PCA for Different Sound Waves')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.legend()
plt.show()

print(f"Original embeddings shape: {embeddings_array.shape}")
print(f"PCA-transformed embeddings shape: {pca_result_multi.shape}")
print(f"Explained variance ratio by principal components: {pca_multi.explained_variance_ratio_}")

**Experiments with Audio Embeddings:**

- Calculate cosine similarlity of different sounds using embedding vectors.  You may take help of Gemini AI tool for this.
- Check PCA variance ratio of waveforms for your choice. See if the correlation match your hypothesis.  

##### **Step 8**:  Execute YAMNet model using LiteRT runtime

In [ ]:
!pip install -q ai-edge-litert

In [ ]:
!wget -O sound-classifier.tflite -q https://storage.googleapis.com/mediapipe-models/audio_classifier/yamnet/float32/1/yamnet.tflite

In [ ]:
from ai_edge_litert.interpreter import Interpreter

import tensorflow as tf
import numpy as np
import zipfile

model_path = ('sound-classifier.tflite')
interpreter = Interpreter(model_path)

input_details = interpreter.get_input_details()
waveform_input_index = input_details[0]['index']
output_details = interpreter.get_output_details()
scores_output_index = output_details[0]['index']

# Input: 0.975 seconds of silence as mono 16 kHz waveform samples.
waveform = np.zeros(int(round(0.975 * 16000)), dtype=np.float32)
#waveform = np.zeros(int(round(3.0 * 16000)), dtype=np.float32)
print(waveform.shape)  # Should print (15600,)

interpreter.resize_tensor_input(waveform_input_index, [waveform.size], strict=True)
interpreter.allocate_tensors()
interpreter.set_tensor(waveform_input_index, waveform)
interpreter.invoke()
scores = interpreter.get_tensor(scores_output_index)
print(scores.shape)  # Should print (1, 521)

top_class_index = scores.argmax()
labels_file = zipfile.ZipFile(model_path).open('yamnet_label_list.txt')
labels = [l.decode('utf-8').strip() for l in labels_file.readlines()]
print(len(labels))  # Should print 521
print(labels[top_class_index])  # Should print 'Silence'.

## **B.  Vision Transformer (ViT) model and comparison with CNN model**

##### **Step 0**:  Install needed Python packages
- **Transformers** (Maintained by Hugging Face. It provides a unified API for downloading, training, and deploying thousands of pretrained models across multiple modalities, including text, vision, and audio.)
- **torch** (core Python package for PyTorch)

In [ ]:
!pip install -q transformers torch

##### **Step 1**: Inferencing using  Vision Transformer (ViT)

ViT used:  vit-base-patch16-224
- vit: Vision Transformer architecture.
- base: Base-sized model parameters (~86 million total parameters).
- patch16: Images are divided into non-overlapping patches of (16x16) pixels.
- 224: Standard expected input resolution of  (224x224) pixels.


In [ ]:
from transformers import pipeline
from PIL import Image
import requests

# Load the ViT pipeline
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

# Load an image from COCO dataset(e.g., a cat)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Predict
results = classifier(image)
for result in results:
    print(f"{result['label']}: {round(result['score'], 4)}")

##### **Step 2:**

To compare the working of a CNN based model and Vision Tranformer(ViT) model, we will use both models on a standard and obfuscated image.  For this we,

- Create a obfuscated image from moving around blocks of pixels
- Check the efficacy of ViT model and a ResNet model to correctly classify the image
- Start with installing LiteRT runtime and converting CNN and ViT models to Lite

The steps used for this example are shown as an ASCII diagram.

```text
      [ INPUT IMAGE ]
             |
             v
    +-----------------+
    |  Pre-processing | (Resize to 224x224, BGR -> RGB)
    +-----------------+
             |
      _______|_______
     |               |
     v               v
 [ PATH A ]      [ PATH B ]
  Original       Obfuscated
   Image           Image
     |               |
     |        +--------------+
     |        | Patch & Mix  | (Split into 56px patches,
     |        |   Shuffle    |  Reconstruct scrambled)
     |        +--------------+
     |               |
     v               v
+-----------------------------+
|   TFLite Inference Engine   |
+-----------------------------+
|                             |
|  [ RESNET-18 (CNN) ]        | --> Result: Likely Fails on Path B
|    (Relies on Local         |              (Shapes broken)
|     Connectivity)           |
|                             |
|  [ ViT-BASE (Transformer) ] | --> Result: Likely Succeeds on Path B
|    (Relies on Global        |              (Attends to patches
|     Attention)              |               anywhere)
+-----------------------------+
             |
             v
      [ PRINT RESULTS ]
   (Compare Argmax Classes)
   ```

**Step 3:**

- Install Keras-Hub and get labels for ImageNet dataset

In [ ]:
!pip install -q keras-hub

In [ ]:
!curl -o imagenet_labels.json https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # Or "tensorflow" or "torch"!

**Step 4:**

- Convert ResNet-18 (CNN) and ViT model to TFLite format
- After conversion these models are saved as resnet18.tflite and vit_base.tflite

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
import keras_hub

# 1. Load the ResNet-18 model from Keras Hub
model = keras_hub.models.ImageClassifier.from_preset(
    "resnet_18_imagenet",
    activation="softmax"
)

# 2. Initialize the converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4. Convert and Save the model
tflite_model = converter.convert()
with open("resnet18.tflite", "wb") as f:
    f.write(tflite_model)

# 1. Load ViT model
print("Available ImageClassifier presets:", keras_hub.models.ImageClassifier.presets.keys())
vit_model = keras_hub.models.ImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet", activation="softmax"
)

converter = tf.lite.TFLiteConverter.from_keras_model(vit_model)

# 2. Enable 'Select TF Ops' for Transformer compatibility
# This allows TFLite to 'fall back' to regular TensorFlow for complex attention ops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Enable standard TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS    # Enable TensorFlow ops fallback
]

# 3. Convert and Save
tflite_vit_model = converter.convert()
with open("vit_base.tflite", "wb") as f:
    f.write(tflite_vit_model)

**Step 5:**

- Create a function to obfuscate image by shuffling patches of a clock image
- Resize image to size 224x224 pixels
- Use LiteRT APIs for inference of both image types

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
import keras_hub
import matplotlib.pyplot as plt
import json

def obfuscate_image(image, patch_size=32):
    """Shuffles patches of the image to obfuscate it."""
    h, w, c = image.shape
    patches = []
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patches.append(image[i:i+patch_size, j:j+patch_size])

    np.random.shuffle(patches)

    # Reconstruct the image
    obfuscated = np.zeros_like(image)
    idx = 0
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            obfuscated[i:i+patch_size, j:j+patch_size] = patches[idx]
            idx += 1
    return obfuscated


def run_tflite_inference(model_path, image):
    """Standard TFLite inference loop."""

    interpreter = Interpreter(model_path=model_path)

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # 1. Determine the target height and width for the input image
    input_shape = input_details[0]['shape']
    model_h, model_w = input_shape[1], input_shape[2]

    target_h, target_w = 224, 224 # Default to 224x224 as it's common for these models
    if model_h > 1 and model_w > 1:
        target_h, target_w = model_h, model_w

    # 2. Resize the interpreter's input tensor based on the determined target dimensions
    # This must be called BEFORE allocate_tensors()
    interpreter.resize_tensor_input(input_details[0]['index'], [1, target_h, target_w, 3])
    interpreter.allocate_tensors() # Allocate tensors after resizing

    # 3. Perform the image resize
    resized = cv2.resize(image, (target_w, target_h))

    # 4. Prepare for the model
    input_data = np.expand_dims(resized, axis=0).astype(np.float32)
    input_data = input_data / 255.0

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    return interpreter.get_tensor(output_details[0]['index'])[0]

# --- Main Logic ---
image_path = "images/clock.jpg"
image = cv2.imread(image_path)

# Load ImageNet labels
with open('imagenet_labels.json', 'r') as f:
    imagenet_labels = json.load(f)

# Check if image was loaded correctly
if image is None:
    print(f"Error: Could not load image from {image_path}")
else:
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize image_rgb to be a multiple of patch_size (56) for obfuscation
    target_dim = 224 # 224 is 4 * 56
    image_rgb_resized = cv2.resize(image_rgb, (target_dim, target_dim))

    # 1. Obfuscate
    patch_size = 56
    obfuscated_rgb = obfuscate_image(image_rgb_resized, patch_size=patch_size)

    # Display side-by-side
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(image_rgb_resized)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(obfuscated_rgb)
    plt.title(f"Obfuscated - {patch_size}px Patches")
    plt.axis("off")

    plt.show()

    # 2. Classify with ResNet (CNN)
    resnet_out = run_tflite_inference("resnet18.tflite", image_rgb_resized )
    resnet_original_class_id = np.argmax(resnet_out)
    print(f"ResNet Original Image Top Class: {resnet_original_class_id} ({imagenet_labels[str(resnet_original_class_id)][1]})")

    resnet_out_ob = run_tflite_inference("resnet18.tflite", obfuscated_rgb)
    resnet_obfuscated_class_id = np.argmax(resnet_out_ob)
    print(f"ResNet Obfuscated Image Top Class: {resnet_obfuscated_class_id} ({imagenet_labels[str(resnet_obfuscated_class_id)][1]})")


    # 3. Classify with ViT (Transformer)
    vit_out = run_tflite_inference("vit_base.tflite", image_rgb_resized)
    vit_original_class_id = np.argmax(vit_out)
    print(f"ViT Original Image Top Class: {vit_original_class_id} ({imagenet_labels[str(vit_original_class_id)][1]})")

    vit_out_g = run_tflite_inference("vit_base.tflite", obfuscated_rgb)
    vit_obfuscated_class_id = np.argmax(vit_out_g)
    print(f"ViT Obfuscated Image Top Class: {vit_obfuscated_class_id} ({imagenet_labels[str(vit_obfuscated_class_id)][1]})")

**Experiments with Vision Transformer Model:**

- Compare the size of CNN and ViT models used above
- Run the above cell multiple times and see if the ViT gets the right answer on the obfuscated image
- What applications do you think are suited for the ViT model?